In [10]:
import os

import polars as pl

import llm_benchmark.utils.dataset as dataset
import llm_benchmark.utils.seshat_requests as seshat_requests
import llm_benchmark.config as config
import llm_benchmark.utils.llm_interface.evaluation_utils as eutils

from typing import Dict, List, Optional, Any, Tuple

module_dir: str = "/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/seshat/main/modules"
cache_dir: str = "/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/seshat"

endpoint_identifiers: Dict[str, str] = seshat_requests.root_search_url(
    "https://seshat-db.com/api/", 
    use_cache=True, 
    cache_url="/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/seshat/cache/seshat_root_url.pkl"
)

polity_mapping: Dict[str, str] = config.polity_mapping

ds: dataset.Dataset = dataset.Dataset(
    identifiers_endpoints=endpoint_identifiers,
    module_dir=module_dir,
    cache_dir=cache_dir,
    override=False,
    ignore_polities=["crisisdb/", "core/references", "core/citations", "core/comments", "core/comment-parts", "core/cliopatria-shapefiles"],
    polity_mapping=polity_mapping
)

Attempting dataset refresh. False
core/macro-regions https://seshat-db.com/api/core/macro-regions/
Loaded module core/macro-regions with 11 entries.
core/regions https://seshat-db.com/api/core/regions/
Loaded module core/regions with 58 entries.
core/ngas https://seshat-db.com/api/core/ngas/
Loaded module core/ngas with 35 entries.
core/polities https://seshat-db.com/api/core/polities/
Loaded module core/polities with 863 entries.
core/capitals https://seshat-db.com/api/core/capitals/
Loaded module core/capitals with 250 entries.
core/nga-polity-relations https://seshat-db.com/api/core/nga-polity-relations/
Loaded module core/nga-polity-relations with 600 entries.
core/sections https://seshat-db.com/api/core/sections/
Loaded module core/sections with 32 entries.
core/subsections https://seshat-db.com/api/core/subsections/
Loaded module core/subsections with 8 entries.
core/variable-hierarchies https://seshat-db.com/api/core/variable-hierarchies/
Loaded module core/variable-hierarchies 

In [2]:
questions_dir = "/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/10_02_2026_1/run 1_qwen/wf/wf_atlatls_questions.csv"

hydrated_df: pl.DataFrame = eutils.hydrate(ds, 
                                           questions_dir,
                                           link_to_dataset=True,)

# Next steps are to implement question answering pipeline, ensuring that the LLM only responds specific prompted questions exactly...
# Then, evaluate the responses -> create a question answering template, for a student alias, then load all repsponses into a single dataframe

In [9]:
import polars as pl
import os

PATH_TO_PLS = "/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/generation/28_05_2026/run_13_Qwen_Qwen2.5-7B/sc/"

def clean(path: str) -> None:
    df = pl.read_csv(path)
    data_new = []
    for row in df.iter_rows(named=True):
        row["output"] = row["output"].split('?')[0] + "?"
        if len(row["output"].split("\n")) > 1:
            row["output"] = ""
        data_new.append(row)
    df = pl.DataFrame(data_new).write_csv(path)

files = os.listdir(PATH_TO_PLS)
for file in files:
    clean(os.path.join(PATH_TO_PLS, file))



In [1]:
from llm_benchmark.utils import benchmark, dataset
from llm_benchmark.utils.benchmark import seshat_setup, hydrate_per_polity
from llm_benchmark.utils.enums import QuestionHydrationOptions


seshat_cache_dir = "/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/seshat"
unhydrated_question_save_path = "/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/generation/28_05_2026/run_13_Qwen_Qwen2.5-7B"
hydrated_question_save_path = "/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/hydrated"
categories_to_evaluate = ['wf', 'sc']

seshat_ds: dataset.Dataset = seshat_setup(seshat_cache_dir=seshat_cache_dir)
    
hydrate_per_polity(dataset=seshat_ds,
                    evaluation_type=QuestionHydrationOptions.PRESENT_ABSENT_UNKNOWN,
                    unhydrated_save_path=unhydrated_question_save_path,
                    hydrated_save_path=hydrated_question_save_path,
                    categories_to_evaluate=categories_to_evaluate)

/Users/apple/miniconda3/envs/llm_benchmark/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Attempting dataset refresh. False
core/macro-regions https://seshat-db.com/api/core/macro-regions/
Ignoring polity core/macro-regions as per configuration.
core/regions https://seshat-db.com/api/core/regions/
Ignoring polity core/regions as per configuration.
core/ngas https://seshat-db.com/api/core/ngas/
Ignoring polity core/ngas as per configuration.
core/polities https://seshat-db.com/api/core/polities/
Ignoring polity core/polities as per configuration.
core/capitals https://seshat-db.com/api/core/capitals/
Ignoring polity core/capitals as per configuration.
core/nga-polity-relations https://seshat-db.com/api/core/nga-polity-relations/
Ignoring polity core/nga-polity-relations as per configuration.
core/sections https://seshat-db.com/api/core/sections/
Ignoring polity core/sections as per configuration.
core/subsections https://seshat-db.com/api/core/subsections/
Ignoring polity core/subsections as per configuration.
core/variable-hierarchies https://seshat-db.com/api/core/variable

Loading hydrated questions for category wf: 100%|██████████| 49/49 [00:00<00:00, 672.86it/s]


Loading hydrated questions for category sc:   8%|▊         | 6/78 [00:00<00:00, 161.90it/s]

Hydrated dataframe written to /Users/apple/Documents/github/neurips_llms/llm-bechmark/db/hydrated/sc_hydrated/sc_polity-territories_questions.csv
Hydrated dataframe written to /Users/apple/Documents/github/neurips_llms/llm-bechmark/db/hydrated/sc_hydrated/sc_polity-populations_questions.csv
Hydrated dataframe written to /Users/apple/Documents/github/neurips_llms/llm-bechmark/db/hydrated/sc_hydrated/sc_population-of-the-largest-settlements_questions.csv
Hydrated dataframe written to /Users/apple/Documents/github/neurips_llms/llm-bechmark/db/hydrated/sc_hydrated/sc_settlement-hierarchies_questions.csv
Hydrated dataframe written to /Users/apple/Documents/github/neurips_llms/llm-bechmark/db/hydrated/sc_hydrated/sc_administrative-levels_questions.csv


TypeError: argument of type 'NoneType' is not a container or iterable